# 🛡️ Glu-Stock: 02_SIGNAL_INFERENCE
**Phase**: Ensemble Intelligence (RF + CNN)

This notebook retrieves candidates from the Firebase `research` queue, performs dual-brain inference, and pushes high-conviction signals to the `signals` queue.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib tensorflow

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase & Secrets)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
from firebase_admin import credentials, db
from datetime import datetime
from typing import List, Dict, Any, Optional
from kaggle_secrets import UserSecretsClient
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON"))
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")

    def get_and_clear_queue(self, queue_name: str) -> List[Any]:
        ref = self.root_ref.child(f"task_queue/{queue_name}")
        tasks = ref.get()
        if not tasks: return []
        ref.delete()
        return list(tasks.values())

    def push_task(self, queue_name: str, data: Any):
        self.root_ref.child(f"task_queue/{queue_name}").push(data)
        
    def log_event(self, phase, details):
        self.root_ref.child("history").push({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Predictors & Agent)
class MLPredictor:
    def __init__(self, model_dir):
        path = os.path.join(model_dir, "glu_brain_v1.joblib")
        brain = joblib.load(path)
        self.model = brain.get("model")
        self.features = brain.get("features", [])

    def predict_proba(self, df): 
        try: return float(self.model.predict_proba(df[self.features].tail(1))[0][1])
        except: return 0.5

class CNNPredictor:
    def __init__(self, model_dir):
        self.interpreters = {}
        for h in ["daily_t2"]: # Simplified for notebook
            path = os.path.join(model_dir, f"cnn_{h}.tflite")
            if os.path.exists(path):
                self.interpreters[h] = tflite.Interpreter(model_path=path)
                self.interpreters[h].allocate_tensors()
                
    def predict(self, df):
        try:
            interpreter = self.interpreters["daily_t2"]
            data = df[['Open', 'High', 'Low', 'Close', 'Volume']].tail(30).values
            data = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0) + 1e-7)
            input_data = np.expand_dims(data.astype(np.float32), axis=0)
            interpreter.set_tensor(interpreter.get_input_details()[0]['index'], input_data)
            interpreter.invoke()
            return float(interpreter.get_tensor(interpreter.get_output_details()[0]['index'])[0][1])
        except: return 0.5

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_inference():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    model_dir = "/kaggle/input/glustock-brains"
    if not os.path.exists(model_dir): 
        print("❌ Models missing! Please add 'glustock-brains' dataset."); return
        
    candidates = fb.get_and_clear_queue("research")
    if not candidates: print("📭 Queue empty."); return

    ml = MLPredictor(model_dir)
    cnn = CNNPredictor(model_dir)
    signals = {}

    for ticker_list in candidates:
        for ticker in ticker_list:
            print(f"🔬 Analyzing {ticker}...")
            df = yf.download(ticker, period="60d", interval="1d", progress=False)
            # Simple technical feature gen for RF
            df['MA20'] = df['Close'].rolling(20).mean()
            df['RSI'] = 50 # Simplified for notebook demo
            
            p1 = ml.predict_proba(df)
            p2 = cnn.predict(df)
            conviction = (p1 + p2) / 2
            
            if conviction > 0.7:
                signals[ticker] = {"conviction": conviction, "price": float(df['Close'].iloc[-1])}
                print(f"🔥 {ticker} BULLISH ({conviction:.2f})")

    if signals:
        fb.push_task("signals", signals)
        fb.log_event("INFERENCE", f"Generated {len(signals)} signals.")

run_inference()